# Model 4 — Binary Logistic Regression dengan Iklan_Keduanya (Joint Advertising)
## Pemilihan Hotel: Novotel (pasca-merger) vs Pullman

Model 3 (dengan `HargaDiff` menggantikan dua variabel harga terpisah) sudah lebih parsimoni
dari Model 2 dengan fit yang setara secara statistik, dan efek harga relatif menjadi signifikan.
Di Model 3, `Iklan_Novotel` signifikan (negatif terhadap peluang memilih Pullman) sedangkan
`Iklan_Pullman` hanya marginal (p ≈ 0.08).

Model 4 menguji ide tambahan: apakah ada efek **khusus** ketika **kedua brand beriklan secara
bersamaan** (misalnya karena "perang iklan" membuat konsumen lebih sensitif terhadap faktor lain,
atau efek clutter yang saling meniadakan), di luar efek masing-masing iklan brand secara terpisah.

Variabel baru: `Iklan_Keduanya` = 1 jika Novotel **dan** Pullman **sama-sama** beriklan
(`Iklan_Novotel == 1` dan `Iklan_Pullman == 1`), dan 0 untuk semua kondisi lain (tidak ada yang
beriklan, atau hanya salah satu yang beriklan).

Model 4 = Model 3 + `Iklan_Keduanya`, sehingga `Iklan_Novotel` dan `Iklan_Pullman` (efek utama
masing-masing brand) tetap dipertahankan di model, dan `Iklan_Keduanya` menangkap efek interaksi
tambahan saat keduanya beriklan bersamaan. Struktur ini konsisten dengan pendekatan Model 2 yang
menambahkan `Income_x_Bisnis` di atas efek utama `Income_JutaRp` dan `TujuanMenginap_Bisnis`.


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
from logit_diagnostics import full_diagnostic_report, validation_report, train_fit_report, train_test_comparison


## 1. Load Data & Bangun Variabel HargaDiff dan Iklan_Keduanya

In [2]:
df = pd.read_csv('Data_Binary_Logit_Pullman-vs-Novotel.csv')

df['BrandPilihan_Pullman'] = (df['BrandPilihan'] == 'Pullman').astype(int)
df['TujuanMenginap_Bisnis'] = (df['TujuanMenginap'] == 'Bisnis').astype(int)
df['Income_x_Bisnis'] = df['Income_JutaRp'] * df['TujuanMenginap_Bisnis']

# Variabel Model 3: selisih harga relatif (positif = Pullman lebih mahal drpd Novotel)
df['HargaDiff'] = df['Harga_Pullman'] - df['Harga_Novotel']

# Variabel baru Model 4: kedua brand beriklan bersamaan (1 = ya, 0 = tidak/hanya salah satu)
df['Iklan_Keduanya'] = ((df['Iklan_Novotel'] == 1) & (df['Iklan_Pullman'] == 1)).astype(int)

feature_cols_m1 = ['TujuanMenginap_Bisnis', 'Income_JutaRp', 'Harga_Novotel',
                    'Harga_Pullman', 'Iklan_Novotel', 'Iklan_Pullman']
feature_cols_m2 = feature_cols_m1 + ['Income_x_Bisnis']
feature_cols_m3 = ['TujuanMenginap_Bisnis', 'Income_JutaRp', 'HargaDiff',
                   'Iklan_Novotel', 'Iklan_Pullman', 'Income_x_Bisnis']
feature_cols_m4 = feature_cols_m3 + ['Iklan_Keduanya']

y = df['BrandPilihan_Pullman']
X1 = df[feature_cols_m1]
X2 = df[feature_cols_m2]
X3 = df[feature_cols_m3]
X4 = df[feature_cols_m4]

print('Distribusi Iklan_Keduanya (crosstab vs Iklan_Novotel & Iklan_Pullman):')
print(pd.crosstab([df['Iklan_Novotel'], df['Iklan_Pullman']], df['Iklan_Keduanya']))
print()
df[['Iklan_Novotel', 'Iklan_Pullman', 'Iklan_Keduanya']].sum()


Distribusi Iklan_Keduanya (crosstab vs Iklan_Novotel & Iklan_Pullman):
Iklan_Keduanya                 0   1
Iklan_Novotel Iklan_Pullman         
0             0              324   0
              1               82   0
1             0               70   0
              1                0  24



Iklan_Novotel      94
Iklan_Pullman     106
Iklan_Keduanya     24
dtype: int64

## 2. Split Data (Identik dengan Model 1, 2 & 3)

`random_state=42` yang sama dipakai lagi supaya baris training/testing identik dan
perbandingan keempat model apple-to-apple.


In [3]:
X1_train, X1_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, random_state=42, stratify=y)
X2_train, X2_test, _, _ = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)
X3_train, X3_test, _, _ = train_test_split(X3, y, test_size=0.2, random_state=42, stratify=y)
X4_train, X4_test, _, _ = train_test_split(X4, y, test_size=0.2, random_state=42, stratify=y)

print('Train:', X4_train.shape, ' | Test:', X4_test.shape)


Train: (400, 7)  | Test: (100, 7)


## 3. Estimasi Model 1, Model 2, Model 3 (Ulang, sbg Pembanding), dan Model 4

In [4]:
model1 = sm.Logit(y_train, sm.add_constant(X1_train)).fit(disp=0)
model2 = sm.Logit(y_train, sm.add_constant(X2_train)).fit(disp=0)
model3 = sm.Logit(y_train, sm.add_constant(X3_train)).fit(disp=0)
model4 = sm.Logit(y_train, sm.add_constant(X4_train)).fit(disp=0)

for name, m in [('Model 1', model1), ('Model 2', model2), ('Model 3', model3), ('Model 4', model4)]:
    print(f'{name}: LL={m.llf:.4f} | AIC={m.aic:.2f} | BIC={m.bic:.2f} | McFadden R2={m.prsquared:.4f} | k={int(m.df_model)+1}')


Model 1: LL=-203.7967 | AIC=421.59 | BIC=449.53 | McFadden R2=0.2648 | k=7
Model 2: LL=-166.1577 | AIC=348.32 | BIC=380.25 | McFadden R2=0.4006 | k=8
Model 3: LL=-166.6440 | AIC=347.29 | BIC=375.23 | McFadden R2=0.3989 | k=7
Model 4: LL=-166.1037 | AIC=348.21 | BIC=380.14 | McFadden R2=0.4008 | k=8


## 4. Uji Restriksi: Apakah Iklan_Keduanya Menambah Informasi di Atas Model 3?

Model 3 adalah **restricted version** dari Model 4 — Model 3 memaksa koefisien
`Iklan_Keduanya` = 0. Ini adalah **1 restriksi linier**, jadi bisa diuji dengan LR test
antar-model (df=1):

H0: b_IklanKeduanya = 0 (restriksi valid, Model 3 cukup, efek gabungan iklan tidak diperlukan)
H1: b_IklanKeduanya ≠ 0 (Model 4 diperlukan, ada efek tambahan saat keduanya beriklan bersamaan)


In [5]:
lr_stat_34 = 2 * (model4.llf - model3.llf)
df_diff_34 = model4.df_model - model3.df_model
lr_pvalue_34 = 1 - stats.chi2.cdf(lr_stat_34, df_diff_34)

print(f'LR statistic (Model 4 vs Model 3) : {lr_stat_34:.4f}')
print(f'df                                : {int(df_diff_34)}')
print(f'p-value                           : {lr_pvalue_34:.4f}')

if lr_pvalue_34 > 0.05:
    print('\n-> GAGAL TOLAK H0: Iklan_Keduanya tidak menambah informasi signifikan.')
    print('   Model 3 tetap cukup; efek utama Iklan_Novotel & Iklan_Pullman sudah memadai.')
else:
    print('\n-> TOLAK H0: Iklan_Keduanya signifikan menambah informasi (Model 4 lebih tepat).')


LR statistic (Model 4 vs Model 3) : 1.0807
df                                : 1
p-value                           : 0.2985

-> GAGAL TOLAK H0: Iklan_Keduanya tidak menambah informasi signifikan.
   Model 3 tetap cukup; efek utama Iklan_Novotel & Iklan_Pullman sudah memadai.


In [6]:
comparison = pd.DataFrame({
    'Model 1': [model1.llf, model1.aic, model1.bic, model1.prsquared, int(model1.df_model)+1],
    'Model 2': [model2.llf, model2.aic, model2.bic, model2.prsquared, int(model2.df_model)+1],
    'Model 3': [model3.llf, model3.aic, model3.bic, model3.prsquared, int(model3.df_model)+1],
    'Model 4': [model4.llf, model4.aic, model4.bic, model4.prsquared, int(model4.df_model)+1],
}, index=['Log-Likelihood', 'AIC', 'BIC', 'McFadden R2', 'Jumlah Parameter'])
comparison.round(4)


,Model 1,Model 2,Model 3,Model 4
Log-Likelihood,-203.7967,-166.1577,-166.6440,-166.1037
AIC,421.5933,348.3154,347.2880,348.2073
BIC,449.5336,380.2471,375.2283,380.1390
McFadden R2,0.2648,0.4006,0.3989,0.4008
Jumlah Parameter,7.0000,8.0000,7.0000,8.0000


## 5. Full Diagnostic Report — Model 4 (Training)

In [7]:
model4, results4 = full_diagnostic_report(
    X4_train, y_train,
    feature_names=feature_cols_m4,
    y_name='BrandPilihan_Pullman',
    model_label='Model 4 (HargaDiff + Iklan_Keduanya)',
    labels=('Novotel', 'Pullman'),
)



######################################################################
#  FULL DIAGNOSTIC REPORT (Binary Logit) - Model 4 (HargaDiff + Iklan_Keduanya)
######################################################################

1. PEARSON CORRELATION MATRIX (antar variabel independen)
                       TujuanMenginap_Bisnis  Income_JutaRp  HargaDiff  Iklan_Novotel  Iklan_Pullman  Income_x_Bisnis  Iklan_Keduanya
TujuanMenginap_Bisnis                  1.000          0.050      0.020          0.088          0.037            0.772           0.023
Income_JutaRp                          0.050          1.000      0.015          0.043         -0.001            0.470          -0.004
HargaDiff                              0.020          0.015      1.000          0.110         -0.095            0.006           0.052
Iklan_Novotel                          0.088          0.043      0.110          1.000          0.035            0.108           0.428
Iklan_Pullman                          0.037    

## 6. Ringkasan statsmodels (opsional, untuk cross-check)

In [8]:
print(model4.summary())


                            Logit Regression Results                            
Dep. Variable:     BrandPilihan_Pullman   No. Observations:                  400
Model:                            Logit   Df Residuals:                      392
Method:                             MLE   Df Model:                            7
Date:                  Mon, 21 Sep 2026   Pseudo R-squ.:                  0.4008
Time:                          13:47:26   Log-Likelihood:                -166.10
converged:                         True   LL-Null:                       -277.21
Covariance Type:              nonrobust   LLR p-value:                 2.229e-44
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -5.7643      1.008     -5.721      0.000      -7.739      -3.789
TujuanMenginap_Bisnis     7.5239      1.003      7.500      0.000       5.558     

## 7. Bandingkan Signifikansi Variabel Iklan: Model 3 vs Model 4

In [9]:
iklan_compare = pd.DataFrame({
    'Model 3 (tanpa Iklan_Keduanya)': {
        'Iklan_Novotel (p-value)': model3.pvalues.get('Iklan_Novotel', np.nan),
        'Iklan_Pullman (p-value)': model3.pvalues.get('Iklan_Pullman', np.nan),
        'Iklan_Keduanya (p-value)': np.nan,
    },
    'Model 4 (+Iklan_Keduanya)': {
        'Iklan_Novotel (p-value)': model4.pvalues.get('Iklan_Novotel', np.nan),
        'Iklan_Pullman (p-value)': model4.pvalues.get('Iklan_Pullman', np.nan),
        'Iklan_Keduanya (p-value)': model4.pvalues.get('Iklan_Keduanya', np.nan),
    },
})
iklan_compare.round(4)


,Model 3 (tanpa Iklan_Keduanya),Model 4 (+Iklan_Keduanya)
Iklan_Novotel (p-value),0.0048,0.0028
Iklan_Pullman (p-value),0.0799,0.3214
Iklan_Keduanya (p-value),NaN,0.3038


## 8. Fit di Data Training vs Validasi Hold-Out (20%) — Model 4

In [10]:
train_result4 = train_fit_report(model4, X4_train, y_train, label='Training Set (80%) - Model 4',
                                  labels=('Novotel', 'Pullman'))


FIT MODEL DI DATA TRAINING - Training Set (80%) - Model 4
--- Classification Table (threshold = 0.5) ---
                Pred Novotel  Pred Pullman
Actual Novotel           139            58
Actual Pullman            21           182

N                  : 400
Hit Rate (Akurasi) : 0.8025  (80.25%)
Sensitivity (Recall Pullman) : 0.8966
Specificity (Recall Novotel) : 0.7056
Precision (Pullman)          : 0.7583

Proportional Chance Criterion : 0.5001 (hit rate model sebaiknya > 1.25x ini = 0.6251, aturan Hair et al.)
  -> Model MEMENUHI kriteria (>1.25x chance)

--- Press's Q Statistic ---
Press's Q = 146.4100  (nilai kritis chi2(1, 0.05) = 3.8415, p-value = 0.0000)
  -> Signifikan (Q > 3.841): hit rate model JAUH LEBIH BAIK dari tebakan acak



In [11]:
test_result4 = validation_report(model4, X4_test, y_test, label='Hold-Out / Test Set (20%) - Model 4',
                                  labels=('Novotel', 'Pullman'))


VALIDASI MODEL - Hold-Out / Test Set (20%) - Model 4
--- Classification Table (threshold = 0.5) ---
                Pred Novotel  Pred Pullman
Actual Novotel            31            18
Actual Pullman             9            42

N                  : 100
Hit Rate (Akurasi) : 0.7300  (73.00%)
Sensitivity (Recall Pullman) : 0.8235
Specificity (Recall Novotel) : 0.6327
Precision (Pullman)          : 0.7000

Proportional Chance Criterion : 0.5002 (hit rate model sebaiknya > 1.25x ini = 0.6252, aturan Hair et al.)
  -> Model MEMENUHI kriteria (>1.25x chance)

--- Press's Q Statistic ---
Press's Q = 21.1600  (nilai kritis chi2(1, 0.05) = 3.8415, p-value = 0.0000)
  -> Signifikan (Q > 3.841): hit rate model JAUH LEBIH BAIK dari tebakan acak



In [12]:
train_test_comparison(train_result4, test_result4)


RINGKASAN PERBANDINGAN TRAIN vs TEST
Hit Rate         Train      Test         Gap
                0.8025    0.7300     +0.0725  <- indikasi overfitting



## 9. Ringkasan Perbandingan 4 Model & Keputusan Akhir

In [13]:
def hit_rate_holdout(model, X_tr, X_te, y_te):
    Xtr_c = sm.add_constant(X_tr)
    Xte_c = sm.add_constant(X_te, has_constant='add')[Xtr_c.columns]
    pred = model.predict(Xte_c)
    return (((pred >= 0.5).astype(int)) == y_te).mean()

final_summary = pd.DataFrame({
    'Model 1 (sederhana)': {
        'AIC': model1.aic, 'BIC': model1.bic, 'McFadden R2': model1.prsquared,
        'Jumlah Parameter': int(model1.df_model) + 1,
        'Hit Rate Hold-Out': hit_rate_holdout(model1, X1_train, X1_test, y_test),
    },
    'Model 2 (+interaksi)': {
        'AIC': model2.aic, 'BIC': model2.bic, 'McFadden R2': model2.prsquared,
        'Jumlah Parameter': int(model2.df_model) + 1,
        'Hit Rate Hold-Out': hit_rate_holdout(model2, X2_train, X2_test, y_test),
    },
    'Model 3 (+HargaDiff)': {
        'AIC': model3.aic, 'BIC': model3.bic, 'McFadden R2': model3.prsquared,
        'Jumlah Parameter': int(model3.df_model) + 1,
        'Hit Rate Hold-Out': hit_rate_holdout(model3, X3_train, X3_test, y_test),
    },
    'Model 4 (+Iklan_Keduanya)': {
        'AIC': model4.aic, 'BIC': model4.bic, 'McFadden R2': model4.prsquared,
        'Jumlah Parameter': int(model4.df_model) + 1,
        'Hit Rate Hold-Out': hit_rate_holdout(model4, X4_train, X4_test, y_test),
    },
}).T
final_summary.round(4)


,AIC,BIC,McFadden R2,Jumlah Parameter,Hit Rate Hold-Out
Model 1 (sederhana),421.5933,449.5336,0.2648,7.0,0.72
Model 2 (+interaksi),348.3154,380.2471,0.4006,8.0,0.73
Model 3 (+HargaDiff),347.2880,375.2283,0.3989,7.0,0.73
Model 4 (+Iklan_Keduanya),348.2073,380.1390,0.4008,8.0,0.73


**Kesimpulan:** Bandingkan AIC/BIC Model 4 terhadap Model 3 dan lihat hasil LR test di Bagian 4:

- Jika LR test **tidak signifikan** (p > 0.05) dan AIC/BIC Model 4 tidak lebih baik dari Model 3,
  maka `Iklan_Keduanya` **tidak perlu ditambahkan** — Model 3 tetap menjadi model final yang paling
  parsimoni, karena efek utama `Iklan_Novotel` dan `Iklan_Pullman` sudah cukup menjelaskan peran iklan.
- Jika LR test **signifikan** (p < 0.05) dan/atau AIC/BIC Model 4 lebih rendah dari Model 3, maka ada
  efek tambahan yang nyata ketika kedua brand beriklan bersamaan (di luar efek masing-masing iklan
  secara terpisah), dan Model 4 menjadi kandidat model final yang lebih lengkap.

*(Catatan: angka-angka aktual di notebook ini baru terisi setelah dijalankan pada environment yang
memiliki file `Data_Binary_Logit_Pullman-vs-Novotel.csv` dan modul `logit_diagnostics.py` — sel-sel
kode di atas belum dieksekusi di sini karena kedua file tersebut tidak diunggah.)*
